# Cell 01 - Load BM25 Index, FAISS Index và corpus metadata

## Mục tiêu cell này
Load lại toàn bộ dữ liệu cần thiết để xây dựng Hybrid Retrieval.

## Vì sao cần làm bước này?
Ở các notebook trước, ta đã có hai retriever riêng:

1. BM25 Retrieval  
   - Tìm theo từ khóa
   - Mạnh khi câu hỏi và tài liệu có nhiều từ giống nhau

2. Dense Retrieval  
   - Tìm theo ngữ nghĩa
   - Mạnh khi câu hỏi và tài liệu khác cách diễn đạt nhưng cùng ý nghĩa

Kết quả validation cho thấy Dense tốt hơn BM25 tổng thể, nhưng BM25 vẫn tìm đúng một số câu mà Dense bỏ sót.  
Vì vậy ở notebook này, ta sẽ kết hợp cả hai để tạo Hybrid Retrieval.

## Giải thích code
Code sẽ:
1. Load `all_chunks.csv` / dense metadata
2. Load BM25 index đã tạo ở notebook 04
3. Load FAISS index đã tạo ở notebook 05
4. Load embedding model `intfloat/multilingual-e5-base`
5. Kiểm tra số lượng chunks, BM25 metadata và FAISS vectors có khớp nhau không

## Output mong đợi
Bạn cần thấy:
- chunks metadata khoảng 91,448 dòng
- FAISS vectors = 91,448
- BM25 index load thành công
- embedding model chạy trên `cuda` nếu GPU khả dụng

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import pickle
import faiss
import torch
from sentence_transformers import SentenceTransformer

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

BM25_DIR = PROJECT / "indexes" / "bm25"
FAISS_DIR = PROJECT / "indexes" / "faiss"
METRIC_DIR = PROJECT / "artifacts" / "metrics"
PRED_DIR = PROJECT / "artifacts" / "predictions"
PROMPT_DIR = PROJECT / "artifacts" / "prompts"

METRIC_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

bm25_index_path = BM25_DIR / "bm25_index.pkl"
bm25_metadata_path = BM25_DIR / "bm25_metadata.csv"
faiss_index_path = FAISS_DIR / "faiss_index.index"
dense_metadata_path = FAISS_DIR / "dense_metadata.csv"

with open(bm25_index_path, "rb") as f:
    bm25_index = pickle.load(f)

chunks_df = pd.read_csv(dense_metadata_path, low_memory=False)
bm25_metadata_df = pd.read_csv(bm25_metadata_path, low_memory=False)
faiss_index = faiss.read_index(str(faiss_index_path))

id_cols = ["corpus_id", "chunk_id", "parent_id", "source_type", "title", "source_path", "language"]

for col in id_cols:
    chunks_df[col] = chunks_df[col].fillna("").astype(str)
    bm25_metadata_df[col] = bm25_metadata_df[col].fillna("").astype(str)

chunks_df["retrieval_text"] = chunks_df["retrieval_text"].fillna("").astype(str)
chunks_df["chunk_text"] = chunks_df["chunk_text"].fillna("").astype(str)

device = "cuda" if torch.cuda.is_available() else "cpu"
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

print("Project:", PROJECT)
print("Chunks metadata:", chunks_df.shape)
print("BM25 metadata:", bm25_metadata_df.shape)
print("FAISS vectors:", faiss_index.ntotal)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Device:", device)

assert len(chunks_df) == len(bm25_metadata_df), "BM25 metadata và dense metadata không khớp số dòng."
assert len(chunks_df) == faiss_index.ntotal, "Metadata và FAISS index không khớp số lượng."

W0510 01:58:52.211000 22424 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Project: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag
Chunks metadata: (91448, 11)
BM25 metadata: (91448, 11)
FAISS vectors: 91448
Embedding model: intfloat/multilingual-e5-base
Device: cuda


# Cell 02 - Định nghĩa hàm BM25 Retrieval và Dense Retrieval

## Mục tiêu cell này
Tạo hai hàm truy xuất riêng:
- `bm25_search_chunks()`: tìm kiếm theo từ khóa bằng BM25
- `dense_search_chunks()`: tìm kiếm theo ngữ nghĩa bằng FAISS Dense Index

## Vì sao cần làm bước này?
Hybrid RAG cần kết hợp hai kiểu tìm kiếm:

1. BM25 Retrieval  
   Phù hợp khi câu hỏi và tài liệu có nhiều từ khóa giống nhau.

2. Dense Retrieval  
   Phù hợp khi câu hỏi và tài liệu khác cách diễn đạt nhưng cùng ý nghĩa.

Trước khi gộp hai kết quả, ta cần có hàm lấy kết quả riêng từ từng retriever.

## Giải thích code
Code sẽ:
1. Định nghĩa tokenizer cho BM25
2. Tạo hàm BM25 search trả về top-k chunks
3. Tạo hàm Dense search trả về top-k chunks
4. Chuẩn hóa output để cả hai hàm đều trả về cùng cấu trúc:
   - rank
   - score
   - retrieval_method
   - corpus_index
   - chunk_id
   - parent_id
   - source_type
   - title
   - chunk_text

## Output mong đợi
Bạn cần thấy BM25 và Dense đều trả về kết quả cho cùng một câu hỏi pháp luật tiếng Việt.

In [2]:
import re

def bm25_tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"[a-zA-ZÀ-ỹ0-9]+", text)
    return [tok for tok in tokens if len(tok) > 1]


def bm25_search_chunks(query, top_k=10, source_filter=None):
    query_tokens = bm25_tokenize(query)
    scores = bm25_index.get_scores(query_tokens)

    if source_filter is not None:
        candidate_indices = chunks_df.index[
            chunks_df["source_type"].isin(source_filter)
        ].to_numpy()
    else:
        candidate_indices = np.arange(len(chunks_df))

    candidate_scores = scores[candidate_indices]
    k = min(top_k, len(candidate_indices))

    top_local = np.argpartition(candidate_scores, -k)[-k:]
    top_local = top_local[np.argsort(candidate_scores[top_local])[::-1]]
    top_indices = candidate_indices[top_local]

    rows = []

    for rank, idx in enumerate(top_indices, start=1):
        row = chunks_df.iloc[idx]

        rows.append({
            "rank": rank,
            "score": float(scores[idx]),
            "retrieval_method": "bm25",
            "corpus_index": int(idx),
            "corpus_id": row["corpus_id"],
            "chunk_id": row["chunk_id"],
            "parent_id": row["parent_id"],
            "source_type": row["source_type"],
            "title": row["title"],
            "language": row["language"],
            "chunk_text": row["chunk_text"]
        })

    return pd.DataFrame(rows)


def dense_search_chunks(query, top_k=10, source_filter=None, search_k=500, max_search_k=None):
    if max_search_k is None:
        max_search_k = len(chunks_df)

    query_embedding = embedding_model.encode(
        ["query: " + str(query)],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    current_search_k = min(search_k, len(chunks_df))
    final_rows = []

    while current_search_k <= max_search_k:
        scores, indices = faiss_index.search(query_embedding, current_search_k)

        rows = []

        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue

            row = chunks_df.iloc[idx]

            if source_filter is not None and row["source_type"] not in source_filter:
                continue

            rows.append({
                "rank": len(rows) + 1,
                "score": float(score),
                "retrieval_method": "dense",
                "corpus_index": int(idx),
                "corpus_id": row["corpus_id"],
                "chunk_id": row["chunk_id"],
                "parent_id": row["parent_id"],
                "source_type": row["source_type"],
                "title": row["title"],
                "language": row["language"],
                "chunk_text": row["chunk_text"]
            })

            if len(rows) >= top_k:
                final_rows = rows
                break

        if len(final_rows) >= top_k:
            break

        current_search_k *= 2

        if current_search_k > max_search_k:
            final_rows = rows
            break

    return pd.DataFrame(final_rows)


test_query = "Lương thử việc được quy định như thế nào?"

print("BM25 results:")
display(
    bm25_search_chunks(test_query, top_k=5, source_filter=["legal"])[
        ["rank", "score", "retrieval_method", "parent_id", "source_type", "title"]
    ]
)

print("\nDense results:")
display(
    dense_search_chunks(test_query, top_k=5, source_filter=["legal"])[
        ["rank", "score", "retrieval_method", "parent_id", "source_type", "title"]
    ]
)

BM25 results:


,rank,score,retrieval_method,parent_id,source_type,title
0,1,28.138447,bm25,106663,legal,legal_cid_106663
1,2,27.446146,bm25,215579,legal,legal_cid_215579
2,3,27.147801,bm25,75051,legal,legal_cid_75051
3,4,26.905449,bm25,118025,legal,legal_cid_118025
4,5,26.775181,bm25,172654,legal,legal_cid_172654



Dense results:


,rank,score,retrieval_method,parent_id,source_type,title
0,1,0.876999,dense,116782,legal,legal_cid_116782
1,2,0.874766,dense,61953,legal,legal_cid_61953
2,3,0.873931,dense,172654,legal,legal_cid_172654
3,4,0.871711,dense,175106,legal,legal_cid_175106
4,5,0.867452,dense,36069,legal,legal_cid_36069


# Cell 03 - Tạo Hybrid Retrieval bằng Reciprocal Rank Fusion

## Mục tiêu cell này
Kết hợp kết quả của BM25 Retrieval và Dense Retrieval thành một danh sách kết quả chung.

## Vì sao cần làm bước này?
BM25 và Dense có điểm mạnh khác nhau:

- BM25 tìm tốt khi câu hỏi và tài liệu trùng từ khóa.
- Dense tìm tốt khi câu hỏi và tài liệu gần nghĩa nhưng không trùng từ khóa hoàn toàn.

Trong notebook 05, ta thấy:
- Dense tìm đúng nhiều câu mà BM25 bỏ sót.
- BM25 vẫn tìm đúng một số câu mà Dense bỏ sót.

Vì vậy, thay vì chọn một trong hai, ta dùng Hybrid Retrieval để kết hợp cả hai.

## Phương pháp sử dụng
Cell này dùng Reciprocal Rank Fusion, viết tắt là RRF.

Ý tưởng đơn giản:
- Tài liệu đứng hạng cao trong BM25 sẽ được cộng điểm.
- Tài liệu đứng hạng cao trong Dense cũng được cộng điểm.
- Nếu một tài liệu xuất hiện tốt ở cả hai retriever, điểm tổng sẽ cao hơn.
- Nếu một tài liệu chỉ xuất hiện ở một retriever nhưng rank rất cao, nó vẫn có cơ hội được giữ lại.

Công thức:

`RRF score = sum(1 / (k + rank))`

Trong đó:
- `rank` là thứ hạng của tài liệu trong từng retriever.
- `k` là hằng số làm mượt, thường dùng 60.

## Giải thích code
Code sẽ:
1. Lấy top candidate từ BM25.
2. Lấy top candidate từ Dense.
3. Gộp kết quả theo `corpus_index`.
4. Tính điểm RRF.
5. Sắp xếp lại theo `rrf_score`.
6. Trả về top-k chunks cuối cùng.

## Output mong đợi
Bạn cần thấy bảng Hybrid results gồm:
- rank
- rrf_score
- bm25_rank
- dense_rank
- parent_id
- source_type
- title

In [3]:
def hybrid_rrf_search_chunks(
    query,
    final_top_k=10,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=None,
    rrf_k=60
):
    bm25_df = bm25_search_chunks(
        query=query,
        top_k=bm25_top_k,
        source_filter=source_filter
    )

    dense_df = dense_search_chunks(
        query=query,
        top_k=dense_top_k,
        source_filter=source_filter,
        search_k=500,
        max_search_k=len(chunks_df)
    )

    fusion = {}

    for _, row in bm25_df.iterrows():
        idx = int(row["corpus_index"])
        fusion.setdefault(idx, {
            "corpus_index": idx,
            "bm25_rank": None,
            "dense_rank": None,
            "bm25_score": None,
            "dense_score": None,
            "rrf_score": 0.0
        })

        fusion[idx]["bm25_rank"] = int(row["rank"])
        fusion[idx]["bm25_score"] = float(row["score"])
        fusion[idx]["rrf_score"] += 1.0 / (rrf_k + int(row["rank"]))

    for _, row in dense_df.iterrows():
        idx = int(row["corpus_index"])
        fusion.setdefault(idx, {
            "corpus_index": idx,
            "bm25_rank": None,
            "dense_rank": None,
            "bm25_score": None,
            "dense_score": None,
            "rrf_score": 0.0
        })

        fusion[idx]["dense_rank"] = int(row["rank"])
        fusion[idx]["dense_score"] = float(row["score"])
        fusion[idx]["rrf_score"] += 1.0 / (rrf_k + int(row["rank"]))

    fused_rows = []

    for item in fusion.values():
        chunk_row = chunks_df.iloc[item["corpus_index"]]

        fused_rows.append({
            "rank": None,
            "rrf_score": item["rrf_score"],
            "bm25_rank": item["bm25_rank"],
            "dense_rank": item["dense_rank"],
            "bm25_score": item["bm25_score"],
            "dense_score": item["dense_score"],
            "corpus_index": item["corpus_index"],
            "corpus_id": chunk_row["corpus_id"],
            "chunk_id": chunk_row["chunk_id"],
            "parent_id": chunk_row["parent_id"],
            "source_type": chunk_row["source_type"],
            "title": chunk_row["title"],
            "language": chunk_row["language"],
            "chunk_text": chunk_row["chunk_text"]
        })

    fused_df = pd.DataFrame(fused_rows)
    fused_df = fused_df.sort_values("rrf_score", ascending=False).head(final_top_k).reset_index(drop=True)
    fused_df["rank"] = range(1, len(fused_df) + 1)

    return fused_df


test_query = "Lương thử việc được quy định như thế nào?"

hybrid_test_df = hybrid_rrf_search_chunks(
    query=test_query,
    final_top_k=10,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=["legal"]
)

display(
    hybrid_test_df[
        [
            "rank",
            "rrf_score",
            "bm25_rank",
            "dense_rank",
            "parent_id",
            "source_type",
            "title"
        ]
    ]
)

,rank,rrf_score,bm25_rank,dense_rank,parent_id,source_type,title
0,1,0.029469,5.0,11.0,172654,legal,legal_cid_172654
1,2,0.027778,3.0,24.0,75051,legal,legal_cid_75051
2,3,0.023158,40.0,16.0,84991,legal,legal_cid_84991
3,4,0.016393,NaN,1.0,116782,legal,legal_cid_116782
4,5,0.016393,1.0,NaN,106663,legal,legal_cid_106663
5,6,0.016129,2.0,NaN,215579,legal,legal_cid_215579
6,7,0.016129,NaN,2.0,61953,legal,legal_cid_61953
7,8,0.015873,NaN,3.0,172654,legal,legal_cid_172654
8,9,0.015625,4.0,NaN,118025,legal,legal_cid_118025
9,10,0.015625,NaN,4.0,175106,legal,legal_cid_175106


# Cell 04 - Tạo hàm Hybrid Retrieval trả về parent_id để đánh giá

## Mục tiêu cell này
Tạo hàm `hybrid_retrieve_parent_ids()` để lấy danh sách `parent_id/cid` không trùng lặp từ kết quả Hybrid Retrieval.

## Vì sao cần làm bước này?
Hybrid Retrieval ở Cell 03 gộp kết quả theo từng chunk.  
Tuy nhiên, ground truth của dataset pháp luật nằm ở mức `cid`, tức là `parent_id`.

Ví dụ:
- chunk `legal_172654_000` có parent_id = 172654
- chunk `legal_172654_001` cũng có parent_id = 172654

Nếu cả hai chunk cùng xuất hiện, khi đánh giá chỉ nên tính là một tài liệu `172654`, không tính lặp.

## Giải thích code
Code sẽ:
1. Gọi `hybrid_rrf_search_chunks()` để lấy nhiều chunk ứng viên
2. Duyệt kết quả theo thứ hạng RRF
3. Quy mỗi chunk về `parent_id`
4. Loại parent_id trùng lặp
5. Trả về top-k parent_id dùng để tính Recall@k, MRR và nDCG

Sau đó test thử trên câu hỏi validation để kiểm tra metric.

## Output mong đợi
Bạn cần thấy:
- câu hỏi mẫu
- relevant cids
- hybrid predicted parent ids
- metric retrieval tương ứng

In [4]:
import math

def recall_at_k(pred_ids, relevant_ids, k):
    return 1.0 if set(pred_ids[:k]) & set(relevant_ids) else 0.0

def mrr_score(pred_ids, relevant_ids):
    relevant_set = set(relevant_ids)
    for rank, pred_id in enumerate(pred_ids, start=1):
        if pred_id in relevant_set:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(pred_ids, relevant_ids, k=10):
    relevant_set = set(relevant_ids)
    dcg = 0.0

    for i, pred_id in enumerate(pred_ids[:k], start=1):
        rel = 1.0 if pred_id in relevant_set else 0.0
        dcg += rel / math.log2(i + 1)

    ideal_hits = min(len(relevant_set), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))

    return dcg / idcg if idcg > 0 else 0.0

def compute_retrieval_metrics(pred_ids, relevant_ids):
    return {
        "recall@1": recall_at_k(pred_ids, relevant_ids, 1),
        "recall@3": recall_at_k(pred_ids, relevant_ids, 3),
        "recall@5": recall_at_k(pred_ids, relevant_ids, 5),
        "recall@10": recall_at_k(pred_ids, relevant_ids, 10),
        "mrr": mrr_score(pred_ids, relevant_ids),
        "ndcg@10": ndcg_at_k(pred_ids, relevant_ids, 10)
    }

def hybrid_retrieve_parent_ids(
    query,
    top_k=10,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=["legal"],
    rrf_k=60
):
    hybrid_df = hybrid_rrf_search_chunks(
        query=query,
        final_top_k=top_k * 5,
        bm25_top_k=bm25_top_k,
        dense_top_k=dense_top_k,
        source_filter=source_filter,
        rrf_k=rrf_k
    )

    parent_ids = []
    seen = set()

    for _, row in hybrid_df.iterrows():
        parent_id = str(row["parent_id"])

        if parent_id not in seen:
            parent_ids.append(parent_id)
            seen.add(parent_id)

        if len(parent_ids) >= top_k:
            break

    return parent_ids

test_row = {
    "question": "Lương thử việc được quy định như thế nào?",
    "relevant_cids": ["61953"]
}

pred_ids = hybrid_retrieve_parent_ids(
    query=test_row["question"],
    top_k=10,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=["legal"]
)

metrics = compute_retrieval_metrics(pred_ids, test_row["relevant_cids"])

print("Question:", test_row["question"])
print("Relevant cids:", test_row["relevant_cids"])
print("Hybrid pred parent_ids:", pred_ids)
print("Metrics:", metrics)

Question: Lương thử việc được quy định như thế nào?
Relevant cids: ['61953']
Hybrid pred parent_ids: ['172654', '75051', '84991', '116782', '106663', '215579', '61953', '118025', '175106', '36069']
Metrics: {'recall@1': 0.0, 'recall@3': 0.0, 'recall@5': 0.0, 'recall@10': 1.0, 'mrr': 0.14285714285714285, 'ndcg@10': 0.3333333333333333}


Kết quả này rất quan trọng: Hybrid RRF chưa chắc luôn tốt hơn Dense trên từng câu riêng lẻ.

Ở câu mẫu:

Relevant cid đúng: 61953
Dense đưa 61953 ở rank 2
Hybrid đưa 61953 xuống rank 7

Lý do là RRF đang cộng điểm từ cả BM25 và Dense. Một số tài liệu khác xuất hiện ở cả hai danh sách nên được cộng điểm cao hơn, làm 61953 bị đẩy xuống. Đây là chuyện bình thường trong Hybrid Retrieval: kết hợp sai cách có thể làm giảm rank của evidence đúng.

Vì Dense của mình đang tốt hơn BM25 tổng thể, ta nên tạo thêm bản Weighted RRF, tức là cho Dense trọng số cao hơn BM25.

# Cell 05 - Tạo Weighted Hybrid Retrieval bằng Weighted RRF

## Mục tiêu cell này
Cải thiện Hybrid Retrieval bằng cách cho Dense Retrieval trọng số cao hơn BM25.

## Vì sao cần làm bước này?
Kết quả validation trước đó cho thấy Dense Retrieval tốt hơn BM25 ở tất cả metric.  
Tuy nhiên, BM25 vẫn có một số câu tìm đúng mà Dense bỏ sót.

Vì vậy, ta không bỏ BM25, nhưng khi gộp bằng RRF nên ưu tiên Dense hơn.

Thay vì dùng RRF thường:

`score = 1 / (k + rank)`

ta dùng Weighted RRF:

`score = bm25_weight / (k + bm25_rank) + dense_weight / (k + dense_rank)`

Nếu `dense_weight > bm25_weight`, các kết quả tốt từ Dense sẽ có ảnh hưởng mạnh hơn.

## Giải thích code
Code sẽ:
1. Tạo hàm `weighted_hybrid_rrf_search_chunks()`
2. Lấy kết quả từ BM25 và Dense
3. Tính weighted RRF score
4. Sắp xếp lại kết quả theo điểm fusion
5. Tạo hàm `weighted_hybrid_retrieve_parent_ids()` để trả về parent_id không trùng
6. Test lại câu hỏi `Lương thử việc được quy định như thế nào?`

## Output mong đợi
Bạn cần thấy `parent_id = 61953` được xếp hạng tốt hơn hoặc ít nhất không bị đẩy xuống quá thấp so với RRF thường.

In [5]:
def weighted_hybrid_rrf_search_chunks(
    query,
    final_top_k=10,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=None,
    rrf_k=60,
    bm25_weight=0.7,
    dense_weight=1.3
):
    bm25_df = bm25_search_chunks(
        query=query,
        top_k=bm25_top_k,
        source_filter=source_filter
    )

    dense_df = dense_search_chunks(
        query=query,
        top_k=dense_top_k,
        source_filter=source_filter,
        search_k=500,
        max_search_k=len(chunks_df)
    )

    fusion = {}

    for _, row in bm25_df.iterrows():
        idx = int(row["corpus_index"])
        fusion.setdefault(idx, {
            "corpus_index": idx,
            "bm25_rank": None,
            "dense_rank": None,
            "bm25_score": None,
            "dense_score": None,
            "rrf_score": 0.0
        })

        fusion[idx]["bm25_rank"] = int(row["rank"])
        fusion[idx]["bm25_score"] = float(row["score"])
        fusion[idx]["rrf_score"] += bm25_weight / (rrf_k + int(row["rank"]))

    for _, row in dense_df.iterrows():
        idx = int(row["corpus_index"])
        fusion.setdefault(idx, {
            "corpus_index": idx,
            "bm25_rank": None,
            "dense_rank": None,
            "bm25_score": None,
            "dense_score": None,
            "rrf_score": 0.0
        })

        fusion[idx]["dense_rank"] = int(row["rank"])
        fusion[idx]["dense_score"] = float(row["score"])
        fusion[idx]["rrf_score"] += dense_weight / (rrf_k + int(row["rank"]))

    fused_rows = []

    for item in fusion.values():
        chunk_row = chunks_df.iloc[item["corpus_index"]]

        fused_rows.append({
            "rank": None,
            "rrf_score": item["rrf_score"],
            "bm25_rank": item["bm25_rank"],
            "dense_rank": item["dense_rank"],
            "bm25_score": item["bm25_score"],
            "dense_score": item["dense_score"],
            "corpus_index": item["corpus_index"],
            "corpus_id": chunk_row["corpus_id"],
            "chunk_id": chunk_row["chunk_id"],
            "parent_id": chunk_row["parent_id"],
            "source_type": chunk_row["source_type"],
            "title": chunk_row["title"],
            "language": chunk_row["language"],
            "chunk_text": chunk_row["chunk_text"]
        })

    fused_df = pd.DataFrame(fused_rows)
    fused_df = fused_df.sort_values("rrf_score", ascending=False).head(final_top_k).reset_index(drop=True)
    fused_df["rank"] = range(1, len(fused_df) + 1)

    return fused_df


def weighted_hybrid_retrieve_parent_ids(
    query,
    top_k=10,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=["legal"],
    rrf_k=60,
    bm25_weight=0.7,
    dense_weight=1.3
):
    hybrid_df = weighted_hybrid_rrf_search_chunks(
        query=query,
        final_top_k=top_k * 5,
        bm25_top_k=bm25_top_k,
        dense_top_k=dense_top_k,
        source_filter=source_filter,
        rrf_k=rrf_k,
        bm25_weight=bm25_weight,
        dense_weight=dense_weight
    )

    parent_ids = []
    seen = set()

    for _, row in hybrid_df.iterrows():
        parent_id = str(row["parent_id"])

        if parent_id not in seen:
            parent_ids.append(parent_id)
            seen.add(parent_id)

        if len(parent_ids) >= top_k:
            break

    return parent_ids


test_question = "Lương thử việc được quy định như thế nào?"
test_relevant = ["61953"]

weighted_hybrid_df = weighted_hybrid_rrf_search_chunks(
    query=test_question,
    final_top_k=15,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=["legal"],
    bm25_weight=0.7,
    dense_weight=1.3
)

weighted_pred_ids = weighted_hybrid_retrieve_parent_ids(
    query=test_question,
    top_k=10,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=["legal"],
    bm25_weight=0.7,
    dense_weight=1.3
)

weighted_metrics = compute_retrieval_metrics(weighted_pred_ids, test_relevant)

display(
    weighted_hybrid_df[
        ["rank", "rrf_score", "bm25_rank", "dense_rank", "parent_id", "source_type", "title"]
    ]
)

print("Relevant cids:", test_relevant)
print("Weighted Hybrid pred parent_ids:", weighted_pred_ids)
print("Metrics:", weighted_metrics)

,rank,rrf_score,bm25_rank,dense_rank,parent_id,source_type,title
0,1,0.029079,5.0,11.0,172654,legal,legal_cid_172654
1,2,0.026587,3.0,24.0,75051,legal,legal_cid_75051
2,3,0.024105,40.0,16.0,84991,legal,legal_cid_84991
3,4,0.021311,NaN,1.0,116782,legal,legal_cid_116782
4,5,0.020968,NaN,2.0,61953,legal,legal_cid_61953
5,6,0.020635,NaN,3.0,172654,legal,legal_cid_172654
6,7,0.020313,NaN,4.0,175106,legal,legal_cid_175106
7,8,0.020000,NaN,5.0,36069,legal,legal_cid_36069
8,9,0.019697,NaN,6.0,243931,legal,legal_cid_243931
9,10,0.019403,NaN,7.0,150289,legal,legal_cid_150289


Relevant cids: ['61953']
Weighted Hybrid pred parent_ids: ['172654', '75051', '84991', '116782', '61953', '175106', '36069', '243931', '150289', '107047']
Metrics: {'recall@1': 0.0, 'recall@3': 0.0, 'recall@5': 1.0, 'recall@10': 1.0, 'mrr': 0.2, 'ndcg@10': 0.38685280723454163}


Kết quả này cho thấy Weighted Hybrid đã cải thiện so với RRF thường, nhưng vẫn chưa chắc tốt hơn Dense cho từng câu riêng lẻ.

Với câu:

Lương thử việc được quy định như thế nào?

Kết quả trước đó:

Hybrid RRF thường: cid đúng 61953 ở rank 7
Weighted Hybrid: cid đúng 61953 ở rank 5
Dense riêng: cid đúng 61953 ở rank 2

Nghĩa là tăng trọng số cho Dense đã giúp 61953 đi từ rank 7 lên rank 5. Nhưng vẫn chưa bằng Dense riêng ở câu này.

# Cell 06 - Đánh giá Hybrid RRF và Weighted Hybrid trên validation set

## Mục tiêu cell này
Đánh giá hai phiên bản Hybrid Retrieval trên toàn bộ validation set:
- `Hybrid_RRF`: BM25 + Dense với trọng số bằng nhau
- `Weighted_Hybrid_RRF`: BM25 + Dense nhưng ưu tiên Dense hơn

## Vì sao cần làm bước này?
Một câu hỏi mẫu không đủ để kết luận phương pháp nào tốt hơn.  
Có trường hợp Dense tốt hơn Hybrid, nhưng cũng có trường hợp BM25 cứu được Dense.

Vì vậy cần đánh giá trên toàn bộ `val_strict.csv` để so sánh công bằng với:
- BM25
- Dense_E5
- Hybrid_RRF
- Weighted_Hybrid_RRF

## Giải thích code
Code sẽ:
1. Load validation set
2. Với mỗi câu hỏi, chạy Hybrid RRF thường
3. Với mỗi câu hỏi, chạy Weighted Hybrid RRF
4. Quy kết quả chunk về `parent_id/cid`
5. Tính Recall@1, Recall@3, Recall@5, Recall@10, MRR, nDCG@10
6. Lưu predictions và metrics vào thư mục `artifacts`

## Output mong đợi
Bạn cần thấy bảng metric của:
- `Hybrid_RRF`
- `Weighted_Hybrid_RRF`

In [6]:
from tqdm.auto import tqdm

SPLIT_DIR = PROJECT / "data" / "splits"

val_df = pd.read_csv(SPLIT_DIR / "val_strict.csv")
val_df["relevant_cids"] = val_df["relevant_cids"].apply(json.loads)

def evaluate_retriever_on_val(method_name, retrieve_fn):
    rows = []

    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc=method_name):
        pred_ids = retrieve_fn(row["question"])
        relevant_ids = [str(x) for x in row["relevant_cids"]]
        metrics = compute_retrieval_metrics(pred_ids, relevant_ids)

        rows.append({
            "qid": row["qid"],
            "question": row["question"],
            "relevant_cids": json.dumps(relevant_ids, ensure_ascii=False),
            "pred_parent_ids": json.dumps(pred_ids, ensure_ascii=False),
            **metrics
        })

    results_df = pd.DataFrame(rows)

    summary_df = (
        results_df[
            ["recall@1", "recall@3", "recall@5", "recall@10", "mrr", "ndcg@10"]
        ]
        .mean()
        .reset_index()
    )

    summary_df.columns = ["metric", "value"]
    summary_df["method"] = method_name

    return results_df, summary_df


hybrid_rrf_results_df, hybrid_rrf_summary_df = evaluate_retriever_on_val(
    method_name="Hybrid_RRF",
    retrieve_fn=lambda q: hybrid_retrieve_parent_ids(
        query=q,
        top_k=10,
        bm25_top_k=50,
        dense_top_k=50,
        source_filter=["legal"],
        rrf_k=60
    )
)

weighted_hybrid_results_df, weighted_hybrid_summary_df = evaluate_retriever_on_val(
    method_name="Weighted_Hybrid_RRF",
    retrieve_fn=lambda q: weighted_hybrid_retrieve_parent_ids(
        query=q,
        top_k=10,
        bm25_top_k=50,
        dense_top_k=50,
        source_filter=["legal"],
        rrf_k=60,
        bm25_weight=0.7,
        dense_weight=1.3
    )
)

hybrid_rrf_pred_path = PRED_DIR / "hybrid_rrf_val_predictions.csv"
weighted_hybrid_pred_path = PRED_DIR / "weighted_hybrid_rrf_val_predictions.csv"

hybrid_rrf_metric_path = METRIC_DIR / "hybrid_rrf_val_metrics.csv"
weighted_hybrid_metric_path = METRIC_DIR / "weighted_hybrid_rrf_val_metrics.csv"

hybrid_rrf_results_df.to_csv(hybrid_rrf_pred_path, index=False, encoding="utf-8-sig")
weighted_hybrid_results_df.to_csv(weighted_hybrid_pred_path, index=False, encoding="utf-8-sig")

hybrid_rrf_summary_df.to_csv(hybrid_rrf_metric_path, index=False, encoding="utf-8-sig")
weighted_hybrid_summary_df.to_csv(weighted_hybrid_metric_path, index=False, encoding="utf-8-sig")

print("Đã lưu Hybrid RRF predictions:", hybrid_rrf_pred_path)
print("Đã lưu Weighted Hybrid predictions:", weighted_hybrid_pred_path)
print("Đã lưu Hybrid RRF metrics:", hybrid_rrf_metric_path)
print("Đã lưu Weighted Hybrid metrics:", weighted_hybrid_metric_path)

display(pd.concat([hybrid_rrf_summary_df, weighted_hybrid_summary_df], ignore_index=True))

Hybrid_RRF:   0%|          | 0/8605 [00:00<?, ?it/s]

Weighted_Hybrid_RRF:   0%|          | 0/8605 [00:00<?, ?it/s]

Đã lưu Hybrid RRF predictions: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\predictions\hybrid_rrf_val_predictions.csv
Đã lưu Weighted Hybrid predictions: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\predictions\weighted_hybrid_rrf_val_predictions.csv
Đã lưu Hybrid RRF metrics: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\hybrid_rrf_val_metrics.csv
Đã lưu Weighted Hybrid metrics: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\weighted_hybrid_rrf_val_metrics.csv


,metric,value,method
0,recall@1,0.412435,Hybrid_RRF
1,recall@3,0.618710,Hybrid_RRF
2,recall@5,0.697385,Hybrid_RRF
3,recall@10,0.781174,Hybrid_RRF
4,mrr,0.533238,Hybrid_RRF
5,ndcg@10,0.579235,Hybrid_RRF
6,recall@1,0.422313,Weighted_Hybrid_RRF
7,recall@3,0.625102,Weighted_Hybrid_RRF
8,recall@5,0.708077,Weighted_Hybrid_RRF
9,recall@10,0.785125,Weighted_Hybrid_RRF


kết quả này rất đáng giá vì nó cho thấy:

Hybrid_RRF thường < Dense_E5
Weighted_Hybrid_RRF > Dense_E5 nhẹ

Cụ thể:

Dense_E5 Recall@10              = 0.783614
Weighted_Hybrid_RRF Recall@10   = 0.785125

Tức là Weighted Hybrid có cải thiện, nhưng cải thiện không quá lớn. Điều này hợp lý vì Dense đã rất mạnh rồi, còn BM25 chỉ hỗ trợ thêm một phần nhỏ.

# Cell 07 - So sánh BM25, Dense và Hybrid Retrieval trên validation set

## Mục tiêu cell này
Tạo bảng so sánh tổng hợp giữa các phương pháp retrieval đã chạy:
- BM25
- Dense_E5
- Hybrid_RRF
- Weighted_Hybrid_RRF

## Vì sao cần làm bước này?
Notebook 07 cần chứng minh Hybrid Retrieval có cải thiện hay không so với các baseline trước đó.

Ở notebook 04, BM25 là baseline theo từ khóa.  
Ở notebook 05, Dense_E5 là baseline theo ngữ nghĩa.  
Ở notebook 07, Hybrid_RRF và Weighted_Hybrid_RRF kết hợp cả hai.

Bảng so sánh này sẽ dùng trực tiếp trong phần kết quả thực nghiệm của báo cáo.

## Giải thích code
Code sẽ:
1. Đọc metric BM25 từ notebook 04
2. Đọc metric Dense từ notebook 05
3. Đọc metric Hybrid_RRF và Weighted_Hybrid_RRF từ notebook 07
4. Gộp tất cả vào một bảng
5. Pivot bảng để mỗi phương pháp là một cột
6. Tính phần cải thiện của Weighted Hybrid so với Dense
7. Lưu bảng so sánh vào `artifacts/metrics/retrieval_methods_val_comparison.csv`

## Output mong đợi
Bạn cần thấy bảng có các metric:
- recall@1
- recall@3
- recall@5
- recall@10
- mrr
- ndcg@10

In [7]:
bm25_metrics_df = pd.read_csv(METRIC_DIR / "bm25_val_metrics.csv")
dense_metrics_df = pd.read_csv(METRIC_DIR / "dense_val_metrics.csv")
hybrid_rrf_metrics_df = pd.read_csv(METRIC_DIR / "hybrid_rrf_val_metrics.csv")
weighted_hybrid_metrics_df = pd.read_csv(METRIC_DIR / "weighted_hybrid_rrf_val_metrics.csv")

all_retrieval_metrics_df = pd.concat(
    [
        bm25_metrics_df,
        dense_metrics_df,
        hybrid_rrf_metrics_df,
        weighted_hybrid_metrics_df
    ],
    ignore_index=True
)

comparison_df = all_retrieval_metrics_df.pivot(
    index="metric",
    columns="method",
    values="value"
).reset_index()

comparison_df["weighted_minus_dense"] = (
    comparison_df["Weighted_Hybrid_RRF"] - comparison_df["Dense_E5"]
)

comparison_df["weighted_vs_dense_percent"] = (
    comparison_df["weighted_minus_dense"] / comparison_df["Dense_E5"] * 100
)

for col in comparison_df.columns:
    if col != "metric":
        comparison_df[col] = comparison_df[col].round(6)

comparison_path = METRIC_DIR / "retrieval_methods_val_comparison.csv"
comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", comparison_path)
display(comparison_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\retrieval_methods_val_comparison.csv


method,metric,BM25,Dense_E5,Hybrid_RRF,Weighted_Hybrid_RRF,weighted_minus_dense,weighted_vs_dense_percent
0,mrr,0.453844,0.540115,0.533238,0.541863,0.001747,0.323501
1,ndcg@10,0.497519,0.585338,0.579235,0.586727,0.001389,0.237284
2,recall@1,0.343754,0.421615,0.412435,0.422313,0.000697,0.165380
3,recall@10,0.687391,0.783614,0.781174,0.785125,0.001511,0.192793
4,recall@3,0.528995,0.624869,0.618710,0.625102,0.000232,0.037195
5,recall@5,0.605113,0.700407,0.697385,0.708077,0.007670,1.095072


Kết quả này rất quan trọng: Weighted_Hybrid_RRF đang là phương pháp tốt nhất hiện tại, nhưng mức cải thiện so với Dense không lớn.

Hiểu nhanh:

BM25                : yếu nhất
Dense_E5            : mạnh hơn BM25 rõ ràng
Hybrid_RRF thường   : hơi kém Dense
Weighted_Hybrid_RRF : tốt nhất, nhỉnh hơn Dense nhẹ

Điểm đáng chú ý:

Recall@10:
Dense_E5            = 0.783614
Weighted_Hybrid_RRF = 0.785125

Recall@5:
Dense_E5            = 0.700407
Weighted_Hybrid_RRF = 0.708077

Nghĩa là Weighted Hybrid giúp cải thiện rõ nhất ở Recall@5, tức là phù hợp với RAG vì mình thường đưa top 3–5 context vào prompt.

# Cell 08 - Chọn retrieval method tốt nhất cho Hybrid RAG

## Mục tiêu cell này
Chọn phương pháp retrieval tốt nhất dựa trên kết quả validation.

## Vì sao cần làm bước này?
Sau khi đánh giá BM25, Dense và Hybrid, ta cần chốt phương pháp nào sẽ dùng làm retrieval chính cho các bước tiếp theo.

Kết quả hiện tại cho thấy:
- BM25 là baseline từ khóa.
- Dense_E5 tốt hơn BM25 rõ ràng.
- Hybrid_RRF thường chưa tốt hơn Dense.
- Weighted_Hybrid_RRF tốt nhất tổng thể, đặc biệt cải thiện Recall@5.

Vì RAG thường đưa top-k context vào prompt, Recall@5 là metric rất quan trọng.  
Do đó, ta chọn `Weighted_Hybrid_RRF` làm retriever chính cho notebook 07.

## Giải thích code
Code sẽ:
1. Đọc bảng so sánh các retrieval methods
2. Tìm phương pháp tốt nhất theo `recall@5`
3. Lưu cấu hình best retriever vào file JSON
4. Hiển thị kết quả lựa chọn

## Output mong đợi
Bạn cần thấy best method là `Weighted_Hybrid_RRF`.

In [8]:
comparison_df = pd.read_csv(METRIC_DIR / "retrieval_methods_val_comparison.csv")

metric_to_select = "recall@5"

row = comparison_df[comparison_df["metric"] == metric_to_select].iloc[0]

method_cols = ["BM25", "Dense_E5", "Hybrid_RRF", "Weighted_Hybrid_RRF"]
best_method = max(method_cols, key=lambda col: row[col])
best_value = row[best_method]

best_retriever_config = {
    "selected_by": metric_to_select,
    "best_method": best_method,
    "best_value": float(best_value),
    "reason": "Weighted Hybrid được chọn vì đạt Recall@5 cao nhất trên validation set, phù hợp với RAG top-k context.",
    "config": {
        "retriever": "weighted_hybrid_rrf",
        "bm25_top_k": 50,
        "dense_top_k": 50,
        "rrf_k": 60,
        "bm25_weight": 0.7,
        "dense_weight": 1.3,
        "final_top_k": 5
    }
}

best_config_path = METRIC_DIR / "best_retriever_config.json"

with open(best_config_path, "w", encoding="utf-8") as f:
    json.dump(best_retriever_config, f, ensure_ascii=False, indent=2)

print("Best metric:", metric_to_select)
print("Best method:", best_method)
print("Best value:", round(best_value, 6))
print("Đã lưu:", best_config_path)

display(comparison_df)

Best metric: recall@5
Best method: Weighted_Hybrid_RRF
Best value: 0.708077
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\best_retriever_config.json


,metric,BM25,Dense_E5,Hybrid_RRF,Weighted_Hybrid_RRF,weighted_minus_dense,weighted_vs_dense_percent
0,mrr,0.453844,0.540115,0.533238,0.541863,0.001747,0.323501
1,ndcg@10,0.497519,0.585338,0.579235,0.586727,0.001389,0.237284
2,recall@1,0.343754,0.421615,0.412435,0.422313,0.000697,0.165380
3,recall@10,0.687391,0.783614,0.781174,0.785125,0.001511,0.192793
4,recall@3,0.528995,0.624869,0.618710,0.625102,0.000232,0.037195
5,recall@5,0.605113,0.700407,0.697385,0.708077,0.007670,1.095072


# Cell 09 - Tạo Hybrid RAG pipeline bằng Weighted Hybrid RRF

## Mục tiêu cell này
Tạo pipeline RAG sử dụng best retriever đã chọn: `Weighted_Hybrid_RRF`.

## Vì sao cần làm bước này?
Ở các cell trước, ta đã chứng minh `Weighted_Hybrid_RRF` là phương pháp retrieval tốt nhất trên validation set, đặc biệt theo `Recall@5`.

Bây giờ ta dùng nó để tạo pipeline RAG thực tế:

User question  
→ BM25 Retrieval  
→ Dense Retrieval  
→ Weighted RRF Fusion  
→ Lấy top-k context  
→ Tạo prompt trả lời có nguồn

## Điểm khác với Naive RAG
Naive RAG ở notebook 06 chỉ dùng Dense Retrieval.  
Hybrid RAG ở notebook 07 dùng cả:
- BM25: mạnh về từ khóa
- Dense: mạnh về ngữ nghĩa
- Weighted RRF: gộp kết quả, ưu tiên Dense hơn

## Giải thích code
Code sẽ:
1. Tạo hàm `get_unique_chunks_by_parent()` để tránh lấy nhiều chunk trùng cùng `parent_id`
2. Tạo hàm `hybrid_retrieve_chunks()` dùng Weighted Hybrid RRF
3. Tạo hàm `format_hybrid_context()`
4. Tạo hàm `build_hybrid_rag_prompt()`
5. Tạo hàm `hybrid_rag_pipeline()`
6. Test thử bằng câu hỏi pháp luật tiếng Việt

## Output mong đợi
Bạn cần thấy:
- retrieved sources từ legal corpus
- prompt Hybrid RAG được lưu vào `artifacts/prompts/latest_hybrid_rag_prompt_legal.txt`

In [9]:
def get_unique_chunks_by_parent(df, top_k=5):
    rows = []
    seen = set()

    for _, row in df.iterrows():
        parent_id = str(row["parent_id"])

        if parent_id in seen:
            continue

        rows.append(row)
        seen.add(parent_id)

        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows).reset_index(drop=True)


def hybrid_retrieve_chunks(
    query,
    top_k=5,
    source_filter=None,
    bm25_top_k=50,
    dense_top_k=50,
    rrf_k=60,
    bm25_weight=0.7,
    dense_weight=1.3
):
    candidate_df = weighted_hybrid_rrf_search_chunks(
        query=query,
        final_top_k=top_k * 5,
        bm25_top_k=bm25_top_k,
        dense_top_k=dense_top_k,
        source_filter=source_filter,
        rrf_k=rrf_k,
        bm25_weight=bm25_weight,
        dense_weight=dense_weight
    )

    result_df = get_unique_chunks_by_parent(candidate_df, top_k=top_k)
    result_df["rank"] = range(1, len(result_df) + 1)

    return result_df


def format_hybrid_context(retrieved_df, max_chars_per_chunk=1200):
    blocks = []

    for _, row in retrieved_df.iterrows():
        text = str(row["chunk_text"])[:max_chars_per_chunk]

        block = f"""
[Source {int(row["rank"])}]
source_type: {row["source_type"]}
title: {row["title"]}
parent_id: {row["parent_id"]}
chunk_id: {row["chunk_id"]}
rrf_score: {row["rrf_score"]:.6f}
bm25_rank: {row["bm25_rank"]}
dense_rank: {row["dense_rank"]}

{text}
""".strip()

        blocks.append(block)

    return "\n\n---\n\n".join(blocks)


def build_hybrid_rag_prompt(question, retrieved_df):
    context = format_hybrid_context(retrieved_df)

    prompt = f"""
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Ưu tiên nguồn có liên quan trực tiếp nhất đến câu hỏi.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
{question}

CONTEXT:
{context}

YÊU CẦU OUTPUT:
1. Câu trả lời
2. Nguồn đã dùng
3. Evidence ngắn gọn
""".strip()

    return prompt


def hybrid_rag_pipeline(question, top_k=5, source_filter=None):
    retrieved_df = hybrid_retrieve_chunks(
        query=question,
        top_k=top_k,
        source_filter=source_filter,
        bm25_top_k=50,
        dense_top_k=50,
        rrf_k=60,
        bm25_weight=0.7,
        dense_weight=1.3
    )

    prompt = build_hybrid_rag_prompt(question, retrieved_df)

    return {
        "question": question,
        "retrieved_sources": retrieved_df,
        "prompt": prompt
    }


legal_question = "Lương thử việc được quy định như thế nào?"

hybrid_legal_output = hybrid_rag_pipeline(
    question=legal_question,
    top_k=5,
    source_filter=["legal"]
)

hybrid_legal_prompt_path = PROMPT_DIR / "latest_hybrid_rag_prompt_legal.txt"
hybrid_legal_prompt_path.write_text(hybrid_legal_output["prompt"], encoding="utf-8")

print("QUESTION:")
print(legal_question)

print("\nĐã lưu prompt:", hybrid_legal_prompt_path)

display(
    hybrid_legal_output["retrieved_sources"][
        ["rank", "rrf_score", "bm25_rank", "dense_rank", "source_type", "title", "parent_id", "chunk_id"]
    ]
)

print("\nPROMPT PREVIEW:")
print(hybrid_legal_output["prompt"][:4000])

QUESTION:
Lương thử việc được quy định như thế nào?

Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_hybrid_rag_prompt_legal.txt


,rank,rrf_score,bm25_rank,dense_rank,source_type,title,parent_id,chunk_id
0,1,0.029079,5.0,11.0,legal,legal_cid_172654,172654,legal_172654_000
1,2,0.026587,3.0,24.0,legal,legal_cid_75051,75051,legal_75051_001
2,3,0.024105,40.0,16.0,legal,legal_cid_84991,84991,legal_84991_001
3,4,0.021311,NaN,1.0,legal,legal_cid_116782,116782,legal_116782_000
4,5,0.020968,NaN,2.0,legal,legal_cid_61953,61953,legal_61953_000



PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Ưu tiên nguồn có liên quan trực tiếp nhất đến câu hỏi.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Lương thử việc được quy định như thế nào?

CONTEXT:
[Source 1]
source_type: legal
title: legal_cid_172654
parent_id: 172654
chunk_id: legal_172654_000
rrf_score: 0.029079
bm25_rank: 5.0
dense_rank: 11.0

Về các chế độ bảo hiểm đối với người thử việc giao kết hợp đồng thử việc 
Câu hỏi: Trong thời gian làm việc theo hợp đồng thử việc, NSDLĐ có phải trả thêm cùng lúc với kỳ trả lương một khoản tiền cho NLĐ tương đương với mức đóng bảo hiểm

# Cell 10 - Tạo Hybrid CrossPolicyRAG cho hai nguồn: handbook nội bộ và pháp luật Việt Nam

## Mục tiêu cell này
Tạo pipeline Hybrid RAG cho câu hỏi cần đối chiếu giữa:
- chính sách nội bộ công ty
- quy định pháp luật Việt Nam

## Vì sao cần làm bước này?
Đề tài của mình không chỉ hỏi luật hoặc hỏi handbook riêng lẻ.  
Use case chính là nhân viên/HR/pháp chế hỏi một chính sách nội bộ khi áp dụng tại Việt Nam thì cần lưu ý gì.

Ví dụ:
`Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?`

Hệ thống cần lấy cả:
1. nguồn handbook nội bộ về quản lý thiết bị
2. nguồn pháp luật/tài liệu Việt Nam liên quan đến thiết bị, tài sản, an toàn, bàn giao hoặc quản lý nơi làm việc

## Lưu ý kỹ thuật
Corpus bị lệch kích thước:
- legal có hơn 91,000 chunks
- company_handbook chỉ có 95 chunks

Vì vậy, ta truy xuất riêng từng nguồn rồi mới gộp lại.  
Với company handbook tiếng Anh nhưng câu hỏi có thể bằng tiếng Việt, Dense Retrieval có vai trò rất quan trọng.  
BM25 vẫn được giữ trong Hybrid, nhưng Dense được ưu tiên hơn.

## Giải thích code
Code sẽ:
1. Tạo hàm `dual_source_hybrid_rag_pipeline()`
2. Lấy top-k nguồn từ `company_handbook`
3. Lấy top-k nguồn từ `legal`
4. Gộp hai nhóm source
5. Tạo prompt Hybrid RAG
6. Lưu prompt vào file để dùng cho demo/báo cáo

## Output mong đợi
Bảng retrieved sources phải có cả:
- `company_handbook`
- `legal`

In [10]:
def dual_source_hybrid_rag_pipeline(question, company_top_k=3, legal_top_k=3):
    company_df = hybrid_retrieve_chunks(
        query=question,
        top_k=company_top_k,
        source_filter=["company_handbook"],
        bm25_top_k=50,
        dense_top_k=50,
        rrf_k=60,
        bm25_weight=0.3,
        dense_weight=1.7
    )

    legal_df = hybrid_retrieve_chunks(
        query=question,
        top_k=legal_top_k,
        source_filter=["legal"],
        bm25_top_k=50,
        dense_top_k=50,
        rrf_k=60,
        bm25_weight=0.7,
        dense_weight=1.3
    )

    retrieved_df = pd.concat([company_df, legal_df], ignore_index=True)
    retrieved_df["rank"] = range(1, len(retrieved_df) + 1)

    prompt = build_hybrid_rag_prompt(question, retrieved_df)

    return {
        "question": question,
        "retrieved_sources": retrieved_df,
        "prompt": prompt
    }


cross_question = "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?"

hybrid_cross_output = dual_source_hybrid_rag_pipeline(
    question=cross_question,
    company_top_k=3,
    legal_top_k=3
)

hybrid_cross_prompt_path = PROMPT_DIR / "latest_hybrid_rag_prompt_cross_policy.txt"
hybrid_cross_prompt_path.write_text(hybrid_cross_output["prompt"], encoding="utf-8")

print("QUESTION:")
print(cross_question)

print("\nĐã lưu prompt:", hybrid_cross_prompt_path)

print("\nRetrieved sources:")
display(
    hybrid_cross_output["retrieved_sources"][
        ["rank", "rrf_score", "bm25_rank", "dense_rank", "source_type", "title", "parent_id", "chunk_id"]
    ]
)

print("\nSố lượng theo source_type:")
display(hybrid_cross_output["retrieved_sources"]["source_type"].value_counts().reset_index())

print("\nPROMPT PREVIEW:")
print(hybrid_cross_output["prompt"][:5000])

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_hybrid_rag_prompt_cross_policy.txt

Retrieved sources:


,rank,rrf_score,bm25_rank,dense_rank,source_type,title,parent_id,chunk_id
0,1,0.027869,NaN,1.0,company_handbook,Managing Work Devices,company_0004,company_0004_001
1,2,0.026562,NaN,4.0,company_handbook,Readme,company_0008,company_0008_000
2,3,0.004918,1.0,NaN,company_handbook,Titles For Support,company_0015,company_0015_013
3,4,0.021311,NaN,1.0,legal,legal_cid_56198,56198,legal_56198_001
4,5,0.021149,47.0,29.0,legal,legal_cid_24406,24406,legal_24406_000
5,6,0.020968,NaN,2.0,legal,legal_cid_117969,117969,legal_117969_000



Số lượng theo source_type:


,source_type,count
0,company_handbook,3
1,legal,3



PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Ưu tiên nguồn có liên quan trực tiếp nhất đến câu hỏi.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

CONTEXT:
[Source 1]
source_type: company_handbook
title: Managing Work Devices
parent_id: company_0004
chunk_id: company_0004_001
rrf_score: 0.027869
bm25_rank: nan
dense_rank: 1.0

With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provi

# Cell 11 - Sửa CrossPolicyRAG: Company dùng Dense, Legal dùng Weighted Hybrid

## Mục tiêu cell này
Cải thiện pipeline CrossPolicyRAG để truy xuất nguồn nội bộ công ty chính xác hơn khi câu hỏi bằng tiếng Việt.

## Vì sao cần làm bước này?
Handbook nội bộ 37signals là tiếng Anh, trong khi người dùng Việt Nam thường hỏi bằng tiếng Việt.

BM25 là tìm kiếm theo từ khóa nên khó tìm đúng tài liệu tiếng Anh khi query là tiếng Việt.  
Ngược lại, Dense Retrieval dùng multilingual embedding nên có thể hiểu gần nghĩa giữa tiếng Việt và tiếng Anh.

Vì vậy, pipeline hợp lý hơn là:
- `company_handbook`: dùng Dense Retrieval
- `legal`: dùng Weighted Hybrid Retrieval

Cách này vẫn giữ đúng tinh thần Hybrid RAG, vì nguồn pháp luật vẫn dùng BM25 + Dense, còn nguồn nội bộ dùng Dense để xử lý truy vấn đa ngôn ngữ.

## Giải thích code
Code sẽ:
1. Tạo hàm `dual_source_hybrid_rag_pipeline_v2()`
2. Truy xuất company handbook bằng `dense_search_chunks()`
3. Truy xuất legal corpus bằng `hybrid_retrieve_chunks()`
4. Gộp hai nhóm source lại
5. Tạo prompt Hybrid RAG mới
6. Kiểm tra xem company sources có tập trung vào `Managing Work Devices` hơn không

## Output mong đợi
Company sources nên chủ yếu là `Managing Work Devices`.  
Legal sources vẫn lấy các tài liệu liên quan đến quản lý thiết bị, nơi làm việc hoặc trang thiết bị.

In [11]:
def dense_retrieve_unique_chunks(query, top_k=3, source_filter=None):
    dense_df = dense_search_chunks(
        query=query,
        top_k=top_k * 5,
        source_filter=source_filter,
        search_k=500,
        max_search_k=len(chunks_df)
    )

    unique_df = get_unique_chunks_by_parent(dense_df, top_k=top_k)
    unique_df["rank"] = range(1, len(unique_df) + 1)

    unique_df["rrf_score"] = unique_df["score"]
    unique_df["bm25_rank"] = np.nan
    unique_df["dense_rank"] = unique_df["rank"]

    return unique_df


def dual_source_hybrid_rag_pipeline_v2(question, company_top_k=3, legal_top_k=3):
    company_df = dense_retrieve_unique_chunks(
        query=question,
        top_k=company_top_k,
        source_filter=["company_handbook"]
    )

    legal_df = hybrid_retrieve_chunks(
        query=question,
        top_k=legal_top_k,
        source_filter=["legal"],
        bm25_top_k=50,
        dense_top_k=50,
        rrf_k=60,
        bm25_weight=0.7,
        dense_weight=1.3
    )

    retrieved_df = pd.concat([company_df, legal_df], ignore_index=True)
    retrieved_df["rank"] = range(1, len(retrieved_df) + 1)

    prompt = build_hybrid_rag_prompt(question, retrieved_df)

    return {
        "question": question,
        "retrieved_sources": retrieved_df,
        "prompt": prompt
    }


cross_question = "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?"

hybrid_cross_output_v2 = dual_source_hybrid_rag_pipeline_v2(
    question=cross_question,
    company_top_k=3,
    legal_top_k=3
)

hybrid_cross_prompt_v2_path = PROMPT_DIR / "latest_hybrid_rag_prompt_cross_policy_v2.txt"
hybrid_cross_prompt_v2_path.write_text(hybrid_cross_output_v2["prompt"], encoding="utf-8")

print("QUESTION:")
print(cross_question)

print("\nĐã lưu prompt:", hybrid_cross_prompt_v2_path)

print("\nRetrieved sources:")
display(
    hybrid_cross_output_v2["retrieved_sources"][
        ["rank", "rrf_score", "bm25_rank", "dense_rank", "source_type", "title", "parent_id", "chunk_id"]
    ]
)

print("\nSố lượng theo source_type:")
display(hybrid_cross_output_v2["retrieved_sources"]["source_type"].value_counts().reset_index())

print("\nPROMPT PREVIEW:")
print(hybrid_cross_output_v2["prompt"][:5000])

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_hybrid_rag_prompt_cross_policy_v2.txt

Retrieved sources:


,rank,rrf_score,bm25_rank,dense_rank,source_type,title,parent_id,chunk_id
0,1,0.791058,NaN,1.0,company_handbook,Managing Work Devices,company_0004,company_0004_001
1,2,0.769166,NaN,2.0,company_handbook,Readme,company_0008,company_0008_000
2,3,0.021311,NaN,1.0,legal,legal_cid_56198,56198,legal_56198_001
3,4,0.021149,47.0,29.0,legal,legal_cid_24406,24406,legal_24406_000
4,5,0.020968,NaN,2.0,legal,legal_cid_117969,117969,legal_117969_000



Số lượng theo source_type:


,source_type,count
0,legal,3
1,company_handbook,2



PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Ưu tiên nguồn có liên quan trực tiếp nhất đến câu hỏi.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

CONTEXT:
[Source 1]
source_type: company_handbook
title: Managing Work Devices
parent_id: company_0004
chunk_id: company_0004_001
rrf_score: 0.791058
bm25_rank: nan
dense_rank: 1.0

With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provi

# Cell 12 - Sửa CrossPolicyRAG để lấy nhiều chunks liên quan từ cùng tài liệu nội bộ

## Mục tiêu cell này
Cải thiện truy xuất `company_handbook` để cho phép lấy nhiều chunk từ cùng một tài liệu nếu các chunk đó đều liên quan.

## Vì sao cần làm bước này?
Trong Cell 11, company handbook chỉ giữ một chunk từ `Managing Work Devices`, sau đó chuyển sang `Readme`.  
Điều này xảy ra vì hàm trước đó loại trùng `parent_id`.

Tuy nhiên, với RAG trả lời câu hỏi thực tế, một tài liệu như `Managing Work Devices` có thể có nhiều đoạn quan trọng:
- đoạn nói về thiết bị được cấp
- đoạn nói về bảo mật thiết bị
- đoạn nói về không lưu code/secrets trên thiết bị cá nhân
- đoạn nói về thiết bị mobile/Windows không được quản lý

Vì vậy, khi tạo context cho LLM, ta nên cho phép nhiều chunk từ cùng tài liệu nội bộ nếu chúng liên quan.

## Chiến lược mới
- Company handbook: dùng Dense Retrieval và cho phép nhiều chunk cùng `parent_id`
- Legal corpus: dùng Weighted Hybrid Retrieval và vẫn deduplicate theo `parent_id/cid`

## Output mong đợi
Company sources nên chủ yếu là các chunk thuộc `Managing Work Devices`.

In [12]:
def dense_retrieve_chunks_no_parent_dedup(query, top_k=3, source_filter=None):
    dense_df = dense_search_chunks(
        query=query,
        top_k=top_k,
        source_filter=source_filter,
        search_k=500,
        max_search_k=len(chunks_df)
    ).copy()

    dense_df["rrf_score"] = dense_df["score"]
    dense_df["bm25_rank"] = np.nan
    dense_df["dense_rank"] = dense_df["rank"]

    return dense_df


def dual_source_hybrid_rag_pipeline_v3(question, company_top_k=3, legal_top_k=3):
    company_df = dense_retrieve_chunks_no_parent_dedup(
        query=question,
        top_k=company_top_k,
        source_filter=["company_handbook"]
    )

    legal_df = hybrid_retrieve_chunks(
        query=question,
        top_k=legal_top_k,
        source_filter=["legal"],
        bm25_top_k=50,
        dense_top_k=50,
        rrf_k=60,
        bm25_weight=0.7,
        dense_weight=1.3
    )

    retrieved_df = pd.concat([company_df, legal_df], ignore_index=True)
    retrieved_df["rank"] = range(1, len(retrieved_df) + 1)

    prompt = build_hybrid_rag_prompt(question, retrieved_df)

    return {
        "question": question,
        "retrieved_sources": retrieved_df,
        "prompt": prompt
    }


cross_question = "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?"

hybrid_cross_output_v3 = dual_source_hybrid_rag_pipeline_v3(
    question=cross_question,
    company_top_k=3,
    legal_top_k=3
)

hybrid_cross_prompt_v3_path = PROMPT_DIR / "latest_hybrid_rag_prompt_cross_policy_v3.txt"
hybrid_cross_prompt_v3_path.write_text(hybrid_cross_output_v3["prompt"], encoding="utf-8")

print("QUESTION:")
print(cross_question)

print("\nĐã lưu prompt:", hybrid_cross_prompt_v3_path)

print("\nRetrieved sources:")
display(
    hybrid_cross_output_v3["retrieved_sources"][
        ["rank", "rrf_score", "bm25_rank", "dense_rank", "source_type", "title", "parent_id", "chunk_id"]
    ]
)

print("\nSố lượng theo source_type:")
display(hybrid_cross_output_v3["retrieved_sources"]["source_type"].value_counts().reset_index())

print("\nCompany handbook sources:")
display(
    hybrid_cross_output_v3["retrieved_sources"][
        hybrid_cross_output_v3["retrieved_sources"]["source_type"] == "company_handbook"
    ][["rank", "title", "parent_id", "chunk_id"]]
)

print("\nPROMPT PREVIEW:")
print(hybrid_cross_output_v3["prompt"][:5000])

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_hybrid_rag_prompt_cross_policy_v3.txt

Retrieved sources:


,rank,rrf_score,bm25_rank,dense_rank,source_type,title,parent_id,chunk_id
0,1,0.791058,NaN,1.0,company_handbook,Managing Work Devices,company_0004,company_0004_001
1,2,0.773632,NaN,2.0,company_handbook,Managing Work Devices,company_0004,company_0004_002
2,3,0.772039,NaN,3.0,company_handbook,Managing Work Devices,company_0004,company_0004_000
3,4,0.021311,NaN,1.0,legal,legal_cid_56198,56198,legal_56198_001
4,5,0.021149,47.0,29.0,legal,legal_cid_24406,24406,legal_24406_000
5,6,0.020968,NaN,2.0,legal,legal_cid_117969,117969,legal_117969_000



Số lượng theo source_type:


,source_type,count
0,company_handbook,3
1,legal,3



Company handbook sources:


,rank,title,parent_id,chunk_id
0,1,Managing Work Devices,company_0004,company_0004_001
1,2,Managing Work Devices,company_0004,company_0004_002
2,3,Managing Work Devices,company_0004,company_0004_000



PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Ưu tiên nguồn có liên quan trực tiếp nhất đến câu hỏi.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

CONTEXT:
[Source 1]
source_type: company_handbook
title: Managing Work Devices
parent_id: company_0004
chunk_id: company_0004_001
rrf_score: 0.791058
bm25_rank: nan
dense_rank: 1.0

With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provi

# Cell 13 - Tạo câu trả lời preview cho Hybrid CrossPolicyRAG

## Mục tiêu cell này
Tạo câu trả lời mẫu từ kết quả Hybrid CrossPolicyRAG đã truy xuất ở Cell 12.

## Vì sao cần làm bước này?
Cell 12 đã lấy được hai nhóm nguồn:
- `company_handbook`: chính sách nội bộ về quản lý thiết bị làm việc
- `legal`: tài liệu pháp luật/tài liệu Việt Nam liên quan đến thiết bị, nơi làm việc hoặc quản lý tài sản

Để demo hệ thống dễ hiểu hơn, ta cần hiển thị kết quả cuối theo format:
1. Câu trả lời
2. Nguồn nội bộ đã dùng
3. Nguồn pháp luật đã dùng
4. Evidence ngắn gọn
5. Lưu ý cần HR/pháp chế kiểm tra thêm

Ở bước này, ta vẫn chưa gọi LLM thật.  
Câu trả lời preview được tạo theo template có kiểm soát để kiểm tra output RAG có đủ nguồn và evidence.

## Giải thích code
Code sẽ:
1. Tách source thành `company_handbook` và `legal`
2. Tạo câu trả lời preview bằng tiếng Việt
3. Lưu output vào `hybrid_cross_policy_preview_output.json`
4. Hiển thị answer, sources và evidence

## Output mong đợi
Bạn cần thấy câu trả lời có nhắc đến:
- chính sách thiết bị nội bộ
- bảo mật thiết bị, code/secrets, thiết bị cá nhân
- lưu ý khi áp dụng tại Việt Nam
- nguồn handbook và nguồn legal

In [13]:
def build_hybrid_cross_policy_preview(rag_output, max_evidence_chars=450):
    retrieved_df = rag_output["retrieved_sources"].copy()

    company_df = retrieved_df[retrieved_df["source_type"] == "company_handbook"].copy()
    legal_df = retrieved_df[retrieved_df["source_type"] == "legal"].copy()

    if company_df.empty and legal_df.empty:
        answer = (
            "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. "
            "Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
        )
    elif company_df.empty:
        answer = (
            "Tôi tìm thấy một số nguồn pháp luật/tài liệu Việt Nam liên quan, "
            "nhưng chưa tìm thấy chính sách nội bộ công ty tương ứng. "
            "Vì vậy chưa đủ căn cứ để đối chiếu chính sách nội bộ với quy định Việt Nam."
        )
    elif legal_df.empty:
        answer = (
            "Tôi tìm thấy chính sách nội bộ công ty về quản lý thiết bị làm việc, "
            "nhưng chưa tìm thấy nguồn pháp luật Việt Nam đủ rõ để đối chiếu. "
            "HR/pháp chế nên kiểm tra thêm trước khi áp dụng."
        )
    else:
        answer = (
            "Theo handbook nội bộ, công ty có chính sách quản lý thiết bị làm việc, "
            "bao gồm cấu hình bảo mật cho thiết bị, kiểm soát truy cập vào hệ thống nội bộ, "
            "không lưu code hoặc secrets trên thiết bị cá nhân, và có thể remote wipe thiết bị khi bị mất "
            "hoặc khi nhân viên rời công ty. "
            "Khi áp dụng cho nhân viên Việt Nam, công ty nên đối chiếu thêm với quy định và quy chế liên quan đến "
            "quản lý trang thiết bị, an toàn nơi làm việc, bàn giao thiết bị, cập nhật thông tin thiết bị và trách nhiệm sử dụng tài sản. "
            "Các nguồn legal truy xuất được có liên quan đến thiết bị/nơi làm việc, nhưng chưa đủ để kết luận pháp lý toàn diện. "
            "Vì vậy HR/pháp chế nên kiểm tra thêm hợp đồng lao động, biên bản bàn giao thiết bị, quy chế tài sản và quy định lao động hiện hành trước khi áp dụng."
        )

    sources = []
    evidence = []

    for _, row in retrieved_df.iterrows():
        sources.append({
            "rank": int(row["rank"]),
            "source_type": row["source_type"],
            "title": row["title"],
            "parent_id": row["parent_id"],
            "chunk_id": row["chunk_id"],
            "rrf_score": round(float(row["rrf_score"]), 6),
            "bm25_rank": None if pd.isna(row["bm25_rank"]) else int(row["bm25_rank"]),
            "dense_rank": None if pd.isna(row["dense_rank"]) else int(row["dense_rank"])
        })

        evidence.append({
            "rank": int(row["rank"]),
            "source_type": row["source_type"],
            "title": row["title"],
            "evidence": str(row["chunk_text"])[:max_evidence_chars]
        })

    return {
        "question": rag_output["question"],
        "answer": answer,
        "sources": sources,
        "evidence": evidence
    }


hybrid_cross_preview = build_hybrid_cross_policy_preview(hybrid_cross_output_v3)

hybrid_cross_preview_path = PROMPT_DIR / "hybrid_cross_policy_preview_output.json"

with open(hybrid_cross_preview_path, "w", encoding="utf-8") as f:
    json.dump(hybrid_cross_preview, f, ensure_ascii=False, indent=2)

print("QUESTION:")
print(hybrid_cross_preview["question"])

print("\nANSWER:")
print(hybrid_cross_preview["answer"])

print("\nSOURCES:")
display(pd.DataFrame(hybrid_cross_preview["sources"]))

print("\nEVIDENCE:")
for item in hybrid_cross_preview["evidence"]:
    print(f"\n[Source {item['rank']}] {item['source_type']} | {item['title']}")
    print(item["evidence"])

print("\nĐã lưu:", hybrid_cross_preview_path)

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

ANSWER:
Theo handbook nội bộ, công ty có chính sách quản lý thiết bị làm việc, bao gồm cấu hình bảo mật cho thiết bị, kiểm soát truy cập vào hệ thống nội bộ, không lưu code hoặc secrets trên thiết bị cá nhân, và có thể remote wipe thiết bị khi bị mất hoặc khi nhân viên rời công ty. Khi áp dụng cho nhân viên Việt Nam, công ty nên đối chiếu thêm với quy định và quy chế liên quan đến quản lý trang thiết bị, an toàn nơi làm việc, bàn giao thiết bị, cập nhật thông tin thiết bị và trách nhiệm sử dụng tài sản. Các nguồn legal truy xuất được có liên quan đến thiết bị/nơi làm việc, nhưng chưa đủ để kết luận pháp lý toàn diện. Vì vậy HR/pháp chế nên kiểm tra thêm hợp đồng lao động, biên bản bàn giao thiết bị, quy chế tài sản và quy định lao động hiện hành trước khi áp dụng.

SOURCES:


,rank,source_type,title,parent_id,chunk_id,rrf_score,bm25_rank,dense_rank
0,1,company_handbook,Managing Work Devices,company_0004,company_0004_001,0.791058,NaN,1
1,2,company_handbook,Managing Work Devices,company_0004,company_0004_002,0.773632,NaN,2
2,3,company_handbook,Managing Work Devices,company_0004,company_0004_000,0.772039,NaN,3
3,4,legal,legal_cid_56198,56198,legal_56198_001,0.021311,NaN,1
4,5,legal,legal_cid_24406,24406,legal_24406_000,0.021149,47.0,29
5,6,legal,legal_cid_117969,117969,legal_117969_000,0.020968,NaN,2



EVIDENCE:

[Source 1] company_handbook | Managing Work Devices
With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provide the employee onboarding/offboarding of access to credentials and our Tailscale VPN to control access to internal systems. We don't use a managed process like Kandji.

## Access to code and secrets
Knowing our devices are safe and secure allows us to entrust our work computers with acces

[Source 2] company_handbook | Managing Work Devices
## Mobile devices and Windows
Devices running Android, iOS/iPadOS, and Windows are currently unmanaged. It’s fine to install our BC4 and HEY apps on these devices to access work projects and email, but since they’re unmanaged – and therefore ‘untrusted’ – it’s not okay to store 37signals code or secrets on them. If you're coding or accessing secure systems, you should be doing so on a Kandji-managed Mac or an Omarchy Linux machi

# Cell 14 - Lưu summary và kiểm tra cuối notebook Hybrid RAG

## Mục tiêu cell này
Lưu lại kết quả quan trọng của notebook `07_hybrid_rag.ipynb` và kiểm tra toàn bộ file đầu ra.

## Vì sao cần làm bước này?
Notebook 07 đã xây dựng và đánh giá Hybrid Retrieval, gồm:
- BM25 Retrieval
- Dense Retrieval
- Hybrid RRF
- Weighted Hybrid RRF
- Hybrid CrossPolicyRAG cho hai nguồn dữ liệu

Kết quả quan trọng nhất là `Weighted_Hybrid_RRF` đạt Recall@5 cao nhất trên validation set, nên được chọn làm retriever tốt nhất hiện tại.

Các file summary này sẽ dùng cho:
- báo cáo kết quả thực nghiệm
- so sánh với Reranker RAG ở notebook 08
- demo Streamlit ở notebook 12

## Giải thích code
Code sẽ:
1. Lưu summary của Hybrid RAG vào file JSON
2. Kiểm tra các file metric, prediction, prompt và preview đã tạo
3. Hiển thị bảng so sánh retrieval methods
4. Hiển thị trạng thái file OK/MISSING

## Output mong đợi
Tất cả file quan trọng của notebook 07 cần có trạng thái `OK`.

In [14]:
hybrid_rag_summary = {
    "notebook": "07_hybrid_rag.ipynb",
    "main_goal": "Build and evaluate Hybrid Retrieval using BM25 + Dense Retrieval.",
    "selected_retriever": "Weighted_Hybrid_RRF",
    "selection_metric": "recall@5",
    "best_retriever_config": {
        "bm25_top_k": 50,
        "dense_top_k": 50,
        "rrf_k": 60,
        "bm25_weight": 0.7,
        "dense_weight": 1.3,
        "final_top_k": 5
    },
    "cross_policy_strategy": {
        "company_handbook": "Dense Retrieval, no parent_id deduplication",
        "legal": "Weighted Hybrid RRF with parent_id deduplication"
    },
    "important_outputs": {
        "retrieval_comparison": str(METRIC_DIR / "retrieval_methods_val_comparison.csv"),
        "best_retriever_config": str(METRIC_DIR / "best_retriever_config.json"),
        "hybrid_cross_policy_prompt": str(PROMPT_DIR / "latest_hybrid_rag_prompt_cross_policy_v3.txt"),
        "hybrid_cross_policy_preview": str(PROMPT_DIR / "hybrid_cross_policy_preview_output.json")
    }
}

hybrid_summary_path = METRIC_DIR / "hybrid_rag_summary.json"

with open(hybrid_summary_path, "w", encoding="utf-8") as f:
    json.dump(hybrid_rag_summary, f, ensure_ascii=False, indent=2)

required_hybrid_files = [
    METRIC_DIR / "hybrid_rrf_val_metrics.csv",
    METRIC_DIR / "weighted_hybrid_rrf_val_metrics.csv",
    METRIC_DIR / "retrieval_methods_val_comparison.csv",
    METRIC_DIR / "best_retriever_config.json",
    METRIC_DIR / "hybrid_rag_summary.json",
    PRED_DIR / "hybrid_rrf_val_predictions.csv",
    PRED_DIR / "weighted_hybrid_rrf_val_predictions.csv",
    PROMPT_DIR / "latest_hybrid_rag_prompt_cross_policy_v3.txt",
    PROMPT_DIR / "hybrid_cross_policy_preview_output.json"
]

hybrid_check_df = pd.DataFrame([
    {
        "file": str(path.relative_to(PROJECT)),
        "status": "OK" if path.exists() else "MISSING",
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 0
    }
    for path in required_hybrid_files
])

comparison_df = pd.read_csv(METRIC_DIR / "retrieval_methods_val_comparison.csv")

print("Đã lưu summary:", hybrid_summary_path)

print("\nRetrieval methods comparison:")
display(comparison_df)

print("\nHybrid output files:")
display(hybrid_check_df)

print("Tổng file OK:", (hybrid_check_df["status"] == "OK").sum(), "/", len(hybrid_check_df))

Đã lưu summary: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\hybrid_rag_summary.json

Retrieval methods comparison:


,metric,BM25,Dense_E5,Hybrid_RRF,Weighted_Hybrid_RRF,weighted_minus_dense,weighted_vs_dense_percent
0,mrr,0.453844,0.540115,0.533238,0.541863,0.001747,0.323501
1,ndcg@10,0.497519,0.585338,0.579235,0.586727,0.001389,0.237284
2,recall@1,0.343754,0.421615,0.412435,0.422313,0.000697,0.165380
3,recall@10,0.687391,0.783614,0.781174,0.785125,0.001511,0.192793
4,recall@3,0.528995,0.624869,0.618710,0.625102,0.000232,0.037195
5,recall@5,0.605113,0.700407,0.697385,0.708077,0.007670,1.095072



Hybrid output files:


,file,status,size_mb
0,artifacts\metrics\hybrid_rrf_val_metrics.csv,OK,0.00
1,artifacts\metrics\weighted_hybrid_rrf_val_metr...,OK,0.00
2,artifacts\metrics\retrieval_methods_val_compar...,OK,0.00
3,artifacts\metrics\best_retriever_config.json,OK,0.00
4,artifacts\metrics\hybrid_rag_summary.json,OK,0.00
5,artifacts\predictions\hybrid_rrf_val_predictio...,OK,2.42
6,artifacts\predictions\weighted_hybrid_rrf_val_...,OK,2.42
7,artifacts\prompts\latest_hybrid_rag_prompt_cro...,OK,0.01
8,artifacts\prompts\hybrid_cross_policy_preview_...,OK,0.01


Tổng file OK: 9 / 9




## 1. File 07 làm gì?

File 07 xây bản **Hybrid RAG**, tức là RAG dùng kết hợp:

```text
BM25 Retrieval  +  Dense Retrieval
```

Hiểu đơn giản:

```text
BM25  = tìm theo từ khóa
Dense = tìm theo ý nghĩa
Hybrid = dùng cả hai rồi gộp kết quả
```

Ví dụ người dùng hỏi:

```text
Lương thử việc được quy định như thế nào?
```

BM25 sẽ tìm các đoạn có từ:

```text
lương, thử việc, quy định
```

Dense sẽ tìm các đoạn gần nghĩa như:

```text
tiền lương trong thời gian thử việc
vi phạm quy định về thử việc
hợp đồng thử việc
```

Hybrid cố gắng tận dụng cả hai.

---

## 2. Vì sao cần Hybrid?

Ở file 04 và 05, mình đã có kết quả:

```text
BM25 Recall@10  = 0.6874
Dense Recall@10 = 0.7836
```

Dense tốt hơn BM25 rõ ràng.

Nhưng mình cũng thấy:

```text
BM25 sai nhưng Dense đúng: 1256 câu
BM25 đúng nhưng Dense sai: 428 câu
```

Ý nghĩa:

```text
Dense mạnh hơn tổng thể,
nhưng BM25 vẫn cứu được 428 câu mà Dense bỏ sót.
```

Vì vậy không nên bỏ BM25 hoàn toàn. File 07 thử kết hợp cả hai.

---

## 3. File 07 đã load những gì?

File 07 load lại:

```text
BM25 index từ file 04
FAISS Dense index từ file 05
metadata của 91,448 chunks
embedding model multilingual-e5-base
```

Kết quả của bạn:

```text
Chunks metadata: 91,448
BM25 metadata: 91,448
FAISS vectors: 91,448
Device: cuda
```

Nghĩa là dữ liệu và index đều khớp nhau.

---

## 4. File 07 tạo lại 2 hàm tìm kiếm

### BM25 search

```text
bm25_search_chunks()
```

Tìm theo từ khóa.

### Dense search

```text
dense_search_chunks()
```

Tìm theo ngữ nghĩa bằng FAISS.

Bạn test câu:

```text
Lương thử việc được quy định như thế nào?
```

BM25 và Dense trả về hai danh sách khác nhau. Đây là dấu hiệu tốt, vì hai retriever đang bổ sung cho nhau.

---

## 5. RRF là gì?

File 07 dùng kỹ thuật:

```text
RRF = Reciprocal Rank Fusion
```

Nói đơn giản: RRF gộp kết quả từ BM25 và Dense dựa trên thứ hạng.

Nếu một tài liệu đứng cao trong BM25 hoặc Dense thì được cộng điểm.

Công thức:

```text
RRF score = 1 / (k + rank)
```

Ví dụ:

```text
Một tài liệu đứng rank 1 trong Dense
→ được điểm cao

Một tài liệu đứng rank 5 trong BM25 và rank 11 trong Dense
→ được cộng điểm từ cả hai phía
```

---

## 6. Vì sao RRF thường chưa tốt?

Bạn test câu “lương thử việc”, kết quả:

```text
Dense đưa cid đúng 61953 ở rank 2
Hybrid RRF thường đưa cid đúng 61953 xuống rank 7
```

Tức là Hybrid thường đôi khi làm tài liệu đúng bị tụt hạng.

Lý do: một số tài liệu khác xuất hiện trong cả BM25 và Dense nên được cộng điểm cao hơn, dù chưa chắc đúng nhất.

---

## 7. Weighted Hybrid là gì?

Để sửa vấn đề đó, mình tạo:

```text
Weighted_Hybrid_RRF
```

Tức là vẫn gộp BM25 + Dense, nhưng cho Dense trọng số cao hơn.

Cấu hình:

```text
bm25_weight = 0.7
dense_weight = 1.3
```

Vì Dense đã chứng minh tốt hơn BM25 trên validation, nên mình ưu tiên Dense hơn.

Với câu “lương thử việc”:

```text
Hybrid RRF thường: cid đúng ở rank 7
Weighted Hybrid: cid đúng lên rank 5
Dense riêng: cid đúng ở rank 2
```

Weighted Hybrid cải thiện so với RRF thường, nhưng không phải lúc nào cũng hơn Dense ở từng câu riêng lẻ.

---

## 8. Kết quả đánh giá toàn validation

Bạn đã chạy trên toàn bộ:

```text
8,605 câu validation
```

Kết quả chính:

```text
BM25 Recall@5              = 0.6051
Dense_E5 Recall@5          = 0.7004
Hybrid_RRF Recall@5        = 0.6974
Weighted_Hybrid_RRF Recall@5 = 0.7081
```

Bảng cuối cho thấy:

```text
Weighted_Hybrid_RRF là tốt nhất theo Recall@5
```

Recall@5 quan trọng vì RAG thường chỉ đưa khoảng top 3–5 context vào prompt.

---

## 9. Kết luận thực nghiệm của file 07

Thứ tự hiệu quả hiện tại:

```text
BM25 < Hybrid_RRF thường < Dense_E5 < Weighted_Hybrid_RRF
```

Nhưng lưu ý:

```text
Weighted Hybrid chỉ nhỉnh hơn Dense một chút.
```

Ví dụ:

```text
Dense Recall@10            = 0.783614
Weighted Hybrid Recall@10  = 0.785125
```

Cải thiện nhỏ, nhưng vẫn là cải thiện thật.

---

## 10. Vì sao chọn Weighted_Hybrid_RRF?

Mình chọn theo metric:

```text
Recall@5
```

Kết quả:

```text
Best method: Weighted_Hybrid_RRF
Recall@5 = 0.708077
```

Lý do chọn Recall@5:

```text
Trong RAG, LLM thường chỉ đọc vài context đầu.
Nếu tài liệu đúng nằm trong top 5 thì khả năng trả lời đúng cao hơn.
```

File đã lưu:

```text
artifacts/metrics/best_retriever_config.json
```

---

## 11. File 07 còn tạo Hybrid RAG demo

Sau khi chọn best retriever, mình tạo pipeline:

```text
hybrid_rag_pipeline()
```

Pipeline này làm:

```text
User question
→ BM25 search
→ Dense search
→ Weighted RRF fusion
→ lấy top context
→ tạo prompt RAG
```

---

## 12. CrossPolicyRAG trong file 07

Use case chính của đề tài là:

```text
Nội bộ công ty + pháp luật Việt Nam
```

Ví dụ:

```text
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?
```

Ban đầu Hybrid CrossPolicyRAG lấy company source chưa tốt vì handbook tiếng Anh, câu hỏi tiếng Việt.

Sau đó mình sửa chiến lược:

```text
Company handbook: dùng Dense Retrieval
Legal corpus: dùng Weighted Hybrid RRF
```

Lý do:

```text
Company handbook là tiếng Anh
User hỏi bằng tiếng Việt
BM25 không phù hợp cho truy vấn khác ngôn ngữ
Dense multilingual phù hợp hơn
```

---

## 13. Vì sao có 6 nguồn trong output?

Vì mình thiết kế:

```text
company_top_k = 3
legal_top_k = 3
```

Nên:

```text
3 nguồn nội bộ
+
3 nguồn legal
=
6 nguồn
```

Ba nguồn nội bộ đều là:

```text
Managing Work Devices
```

Đây là tốt, vì nó đúng chính sách thiết bị làm việc.

Ba nguồn legal liên quan đến:

```text
thiết bị
nơi làm việc
quản lý trang thiết bị
cập nhật thiết bị CNTT
```

Nhưng legal source vẫn chưa hoàn hảo.

---

## 14. Output Cell 13 chứng minh gì?

Cell 13 tạo câu trả lời preview:

```text
Theo handbook nội bộ, công ty có chính sách quản lý thiết bị làm việc...
Khi áp dụng cho nhân viên Việt Nam, cần kiểm tra thêm quy chế tài sản, bàn giao thiết bị, hợp đồng lao động...
Các nguồn legal có liên quan nhưng chưa đủ để kết luận toàn diện...
```

Điểm tốt:

```text
Có nguồn nội bộ rõ ràng
Có nguồn legal hỗ trợ
Không kết luận pháp lý quá đà
Có cảnh báo HR/pháp chế kiểm tra thêm
```

Đây là cách trả lời an toàn cho doanh nghiệp.

---

## 15. Hạn chế của file 07

File 07 vẫn có hạn chế:

```text
Legal sources đôi khi còn rộng hoặc hơi lệch.
Hybrid chỉ fusion theo rank, chưa thật sự đọc hiểu từng source.
Chưa có reranker để đánh giá source nào liên quan nhất.
Chưa gọi LLM thật để sinh answer tự nhiên.
```

Ví dụ `legal_cid_24406` nói về thiết bị sinh hoạt cho cơ quan Việt Nam ở nước ngoài, hơi lệch với chính sách thiết bị công ty tư nhân.

---

## 16. Vì sao cần file 08?

File 08 sẽ làm:

```text
Reranker RAG
```

Reranker giống như bước “đọc lại và chấm điểm” các nguồn đã retrieve.

Pipeline sẽ là:

```text
BM25 + Dense lấy nhiều candidates
→ Reranker đọc câu hỏi + từng context
→ xếp lại nguồn nào liên quan nhất
→ đưa top context tốt hơn cho LLM
```

Mục tiêu là sửa hạn chế của file 07:

```text
lọc bớt nguồn legal lệch
đẩy nguồn đúng lên cao hơn
cải thiện chất lượng context
```

---

## Chốt ngắn gọn file 07

File 07 làm được 3 việc lớn:

```text
1. Đánh giá Hybrid Retrieval trên validation.
2. Chọn Weighted_Hybrid_RRF làm retriever tốt nhất hiện tại.
3. Tạo Hybrid CrossPolicyRAG: lấy policy nội bộ + legal Việt Nam để tạo prompt có nguồn.
```

Kết quả quan trọng nhất:

```text
Weighted_Hybrid_RRF đạt Recall@5 = 0.7081
cao hơn BM25 và Dense.
```

Nhưng vì legal sources vẫn chưa đủ sắc, bước tiếp theo file 08 sẽ dùng **Reranker** để xếp hạng lại context tốt hơn.
